In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from typing import Annotated, Dict



class GetDeviceInFovInput(BaseModel):
    isInFov: bool = Field(
        default=True,
        description= """
    - isInFov: If True, returns devices that are within the user's line of sight.
    """
    )
    order: str = Field(
        default="proximity",
        description= """
    - order : The sorting method for devices. Possible values are:
          - "proximity" (closest first, default)
          - "right" (when you need to sort the devices from right to left)
          - "high" (when you need to sort the devices from high to low with  z value)
    """
    )
    range: Optional[float] = Field(
        default=None,
        description="""
    - range : The distance range from user to search for devices (in meters). 
          Optional, default is no range limitation.
    """
    )

    class Config:
        extra = "forbid"  # additionalProperties: false にする





@tool
def getDeviceInFov(params: GetDeviceInFovInput) -> Dict:
    """
    This function retrieves devices that are within the user's line of sight.
    return value is a list of devices in which its order is according to the argument.
    """
    
    try:
        print()
        print("=====================[TOOL] getDevicesInSights================")
        request_body = {
            "isInFov": param.isInFov,
            "order": param.order,
            "range": param.range
            }
        
        print(f"Sending POST request to {base_url}/fov with body: {request_body}")
        response = httpx.post(f"{base_url}/fov", json=request_body)
        
        if response.status_code == 200:
            response_data = response.json()
            received_devices = response_data["devices"]
            if response_data.get("status") == "success":
                print(response_data)
                print("================================================")
                print()
                return response_data
            else:
                return {"status": "error", "message": "Server responded with an error", "details": response_data}
        else:
            return {"status": "error", "message": f"HTTP Error {response.status_code}", "details": response.text}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
class GetDeviceInDirectionInput(BaseModel):
    direction: str = Field(
        default="Front",
        description=  """
    - direction (str): The direction to search for devices. Possible values are:
          - "Front" (devices in front of the user)
          - "Back" (devices behind the user)
          - "Left" (devices to the left of the user)
          - "Right" (devices to the right of the user)
    """
    )
    order: str = Field(
        default="proximity",
        description= """
    - order : The sorting method for devices. Possible values are:
          - "proximity" (closest first, default)
          - "right" (when you need to sort the devices from right to left)
          - "high" (when you need to sort the devices from high to low with  z value)
    """
    )
    range: Optional[float] = Field(
        default=None,
        description="""
    - range : The distance range from user to search for devices (in meters). 
          Optional, default is no range limitation.
    """
    )
@tool
def getDeviceInDirection(
    param : GetDeviceInDirectionInput
) -> Dict:
    """
    This function retrieves devices based on the user's retrieved direction.
    return boolean value indicates whether the devices are successfully received.
    """
    try:
        print()
        print("=====================[TOOL] getDevicesInSights================")
        request_body = {
        "direction" : param.direction,
        "order": param.order ,
        "range" : param.range
    }
        
        
        print(f"Sending POST request to {base_url}/direction with body: {request_body}")
        response = httpx.post(f"{base_url}/direction", json=request_body)
        
        if response.status_code == 200:
            response_data = response.json()
            received_devices = response_data["devices"]
            if response_data.get("status") == "success":
                print(response_data)
                
                print("================================================")
                print()
                return response_data
            else:
                return {"status": "error", "message": "Server responded with an error", "details": response_data}
        else:
            return {"status": "error", "message": f"HTTP Error {response.status_code}", "details": response.text}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [38]:

import paho.mqtt.client as mqtt

class MQTTPublisher:
    def __init__(self, broker_address, port=1883):
        """
        コンストラクタでブローカーに接続します。
        :param broker_address: MQTTブローカーのアドレス
        :param port: MQTTブローカーのポート（デフォルトは1883）
        """
        self.broker_address = broker_address
        self.port = port
        self.client = mqtt.Client()

        # 接続時のコールバック設定
        self.client.on_connect = self.on_connect
        self.client.on_publish = self.on_publish

        # ブローカーに接続
        self.client.connect(self.broker_address, self.port, 60)

    def on_connect(self, client, userdata, flags, rc):
        """
        MQTTブローカーに接続したときのコールバック
        :param client: MQTTクライアントインスタンス
        :param userdata: ユーザーデータ（今回は使用しません）
        :param flags: 接続フラグ（今回は使用しません）
        :param rc: 接続結果
        """
        if rc == 0:
            print("Connected to MQTT broker successfully.")
        else:
            print(f"Failed to connect with result code {rc}")

    def on_publish(self, client, userdata, mid):
        """
        メッセージが公開されたときのコールバック
        :param client: MQTTクライアントインスタンス
        :param userdata: ユーザーデータ（今回は使用しません）
        :param mid: メッセージID
        """
        print(f"Message published with ID: {mid}")

    def send_data(self, topic, payload):
        """
        指定されたトピックにデータを送信します。
        :param topic: MQTTトピック
        :param payload: 送信するデータ
        """
        self.client.loop_start()  # 非同期でメインループを開始
        result = self.client.publish(topic, payload)  # データの送信
        if result.rc != mqtt.MQTT_ERR_SUCCESS:
            print("Error publishing message")
        self.client.loop_stop()  # メインループを停止






In [45]:
import httpx
import os

class DeviceOperator():
    def __init__(self):
        self.mqtt_publisher = MQTTPublisher("localhost", 1883)
        self.request_url = os.getenv("XR_SERVER_API") + "/device/operate"
        
    def send_simulation_request(self, devices):
        print("Sending Operate Request to Test Server.") 
        response = http.post(self.simulation_id, json=devices)
        output = ""
        print("CODE: ", response.status_code)
        if response.status_code == 200:
            output = f"[Status] {response.status_code}\n[Message] {response.json()['message']}"
            print("All Devices Are Operated Successfully.")
        else:
            output = f"[Status] {response.status_code}\n[Message] Failed to Operate Devices."
            print("Failed to Operate Devices.")
        return output



    
    def send_operator(self, devices):
        all_devices = httpx.get("http://localhost:4049/device/get-all").json()
        
        mqtts = []
        switchbots = []
    
        for device in all_devices:
            conn_type = device.get("connection_type")
            if conn_type == "mqtt":
                mqtts.append(device)
            elif conn_type == "switchbot":
                switchbots.append(device)
    
        self.send_mqtt_request(mqtts)
        self.send_switchbot_request(switchbots)
    
        return all_devices


    def send_mqtt_request(self, devices): 
        for device in devices: 
            topic = getattr(device, "topic", None)
            if not topic:
                print(f"Device missing topic: {device}")
                continue
    
            try:
                payload = device.json()
            except AttributeError:
                payload = json.dumps(device)  # 辞書など別形式なら fallback
            self.mqtt_publisher.send_data(topic, payload)

    def send_switchbot_request(self, devices):
        for device in devices:
            switchbot_id = device.get("topic")
            if not switchbot_id:
                print(f"Missing device_id in SwitchBot device: {device}")
                continue
    
            try:
                # DeviceControlDataを生成（ID, state, intensity, color などを仮定）
                control_data = DeviceControlData(
                    id=switchbot_id,
                    state=device.get("state", True),  # デフォルト ON
                    intensity=device.get("intensity", 100),  # デフォルト最大
                    color=RGBColor(**device.get("color", {"r": 255, "g": 255, "b": 255}))  # デフォルト白
                )
            except Exception as e:
                print(f"Failed to create control data for SwitchBot: {e}")
                continue
    
            self.operate_switchbot(control_data)

    

In [30]:
t = DeviceOperator() 
res = t.send_operator([])
all_devices = res.json()



mqtts = [device for device in all_devices if device["connection_type"] == "mqtt"]
switchbot  = [device for device in all_devices if device["connection_type"] == "switchbot"]

switchbot

[{'device_id': 'dev123',
  'anchor_id': 'anchor456',
  'topic': 'switchbot/test',
  'device_type': 'light',
  'device_name': 'Ceiling Light',
  'description': 'Main ceiling light in the living room',
  'device_position': {'x': 1.5, 'y': 2, 'z': -0.5},
  'connection_type': 'switchbot'}]

In [4]:
import httpx




url = "http://127.0.0.1:8800/llm_agent"

data = {
"llm_message" : "目の前の電気を点けてください", 
    "task_id" : "1"
}



res = httpx.post(url, json=data, timeout=60)

In [6]:
res.content

b'{"output":"It seems like there are no devices currently identified as being \\"in front of you\\" according to the available data. Please ensure that the devices are correctly aligned with your field of view or provide more specific instructions so I can assist you better."}'

In [5]:
import httpx

res = httpx.get("http://localhost:4049/device/get-all")
res.json()

[{'device_id': 'dev123',
  'anchor_id': 'anchor456',
  'topic': 'home/livingroom/light1',
  'device_type': 'light',
  'device_name': 'Ceiling Light',
  'description': 'Main ceiling light in the living room',
  'device_position': {'x': 1.5, 'y': 2, 'z': -0.5},
  'connection_type': 'mqtt'}]

In [7]:
res = httpx.get("http://localhost:4049/device/get-all")
data = res.json()  # おそらくリスト形式のJSON（例: List[Dict]）

filter_ids = ["dev123"]

# フィルタリング
filtered_devices = [device for device in data if device["device_id"] in filter_ids]

print(filtered_devices)



[{'device_id': 'dev123', 'anchor_id': 'anchor456', 'topic': 'home/livingroom/light1', 'device_type': 'light', 'device_name': 'Ceiling Light', 'description': 'Main ceiling light in the living room', 'device_position': {'x': 1.5, 'y': 2, 'z': -0.5}, 'connection_type': 'mqtt'}]


In [ ]:

@tool
def operateDevice(   devices: List[DeviceControlData]) -> str:
    """
    This function operates devices based on provided control data.
    Example input:
    [
        {
            "id": "test_light_id",
            "state": true,
            "intensity": 100,
            "color": {"r": 255, "g": 255, "b": 255}  # default is white
        },
         {
            "id": "test_curtain_id",
            "state": true,
            "intensity": 100,
            "color": {"r": 255, "g": 255, "b": 255}
        }
    ]


    **For curtain, 0 = open, 100 = close    
    """


    try:

        print()
        print("=====================[OPERATOR TOOL] operateDevice=====================")
        # デバイスデータを取得
        convert_data =  [device.dict() for device in devices]
        response = testOperator.send_operate_request(convert_data)
        print("RESPONSE: ", response)
        return  f"RESULT: {response}"
    except Exception as e:
        print("ERROR OCCURRED DURING OPERATION TOOL: ", e)
        return f"エラーが発生しました: {e}"



